In [ ]:
import pandas as pd
from fredapi import Fred
from settings import fred_api_key

In [ ]:
# Import df from csv
raw_df = pd.read_csv('raw_data.csv')

In [ ]:
## FROM RAW DATA TO df_y

# Load and clean columns
df = raw_df.iloc[1:].copy()
df.columns = df.columns.str.strip()

# Melt to long format: 'Capital IQ Information' holds info, others are tickers
df_long = df.melt(id_vars=['Capital IQ Information'], var_name='Ticker', value_name='Value')

# Clean numeric values
df_long['Value'] = df_long['Value'].str.replace(',', '').str.strip()
df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')

# Filter only median revenue consensus estimates
mask = df_long['Capital IQ Information'].str.contains('Revenue Median Consensus Estimate', na=False)
df_median = df_long[mask].copy()

# Extract forecast year
df_median['Year'] = df_median['Capital IQ Information'].str.extract(r'CY (\d{4})')[0].astype(int)

# Drop unnecessary column and rename for clarity
df_median = df_median.rename(columns={'Value': 'RevenueForecast'})

# Keep only necessary columns and drop missing values
df_y = df_median[['Ticker', 'Year', 'RevenueForecast']].dropna()

print(df_y.head(10))


In [ ]:
import pandas as pd
import numpy as np

# Load raw data
raw_df = pd.read_csv('raw_data.csv')
raw_df.columns = raw_df.columns.str.strip()

# Melt into long format
tickers = raw_df.columns.drop('Capital IQ Information')
df_long = raw_df.melt(id_vars='Capital IQ Information', value_vars=tickers,
                      var_name='Ticker', value_name='Value')

# Clean strings
df_long['Capital IQ Information'] = df_long['Capital IQ Information'].str.strip()
df_long['Value'] = df_long['Value'].astype(str).str.strip()


# Function to convert financial string values, including negatives in parentheses
def convert_financial_value(val):
    if val in ['n/a', 'NA', '', None, np.nan]:
        return np.nan
    val = val.replace(',', '')
    if val.startswith('(') and val.endswith(')'):
        val = '-' + val[1:-1]
    try:
        return float(val)
    except:
        return np.nan


df_long['Value'] = df_long['Value'].apply(convert_financial_value)

# Filter only historical EBITDA and Revenue rows
mask_hist = df_long['Capital IQ Information'].str.contains('Last FY')
df_hist = df_long[mask_hist].copy()

# Extract Year information:
# Suppose your valuation date is CY 2024, so:
# Last FY -> 2023
# Last FY - 1 -> 2022
# Last FY - 2 -> 2021
# ... adjust accordingly

valuation_year = 2025


def extract_hist_year(row):
    if 'Last FY -' in row:
        n = int(row.split('Last FY -')[-1].strip())
        return valuation_year - 1 - n
    elif 'Last FY' in row:
        return valuation_year - 1
    else:
        return np.nan


df_hist['Year'] = df_hist['Capital IQ Information'].apply(extract_hist_year)

# Extract FeatureType (EBITDA or Revenue)
df_hist['FeatureType'] = df_hist['Capital IQ Information'].str.extract(r'(EBITDA|Revenue)')

# Pivot so each feature is a column
df_features_hist = df_hist.pivot_table(index=['Ticker', 'Year'],
                                       columns='FeatureType',
                                       values='Value').reset_index()

# Clean column names
df_features_hist.columns.name = None

print(df_features_hist.head(30))


In [ ]:
# BUILD RISK FREE RATE DF
# --- Set your FRED API key here ---
fred = Fred(api_key=fred_api_key)
gs10 = fred.get_series('GS10', observation_start='2020-01-01', observation_end='2029-12-31')

# Convert to DataFrame
df_gs10 = pd.DataFrame(gs10)
df_gs10.index = pd.to_datetime(df_gs10.index)
df_gs10.columns = ['Risk_Free_Rate']

# Resample to annual average yield
df_risk_free = df_gs10.resample('YE').mean().reset_index()

# Extract Year
df_risk_free['Year'] = df_risk_free['index'].dt.year

# Keep only needed columns and sort
df_risk_free = df_risk_free[['Year', 'Risk_Free_Rate']].sort_values('Year').reset_index(drop=True)

# --- INFLATION DF ----
cpi = fred.get_series('CPIAUCSL', observation_start='2019-12-31', observation_end='2029-12-31')
# Convert to DataFrame
df_cpi = pd.DataFrame(cpi)
df_cpi.index = pd.to_datetime(df_cpi.index)
df_cpi.columns = ['CPI']

# Resample to annual average CPI
df_cpi_annual = df_cpi.resample('YE').mean().reset_index()

# Extract Year
df_cpi_annual['Year'] = df_cpi_annual['index'].dt.year

# Calculate YoY % change in CPI as Inflation Rate
df_cpi_annual['Inflation'] = df_cpi_annual['CPI'].pct_change() * 100

# Keep only Year and Inflation columns, drop first NaN row
df_cpi_final = df_cpi_annual[['Year', 'Inflation']].dropna().reset_index(drop=True)

# ----- GDP DF -----
# Fetch GDP data from FRED
gdp = fred.get_series('GDP', observation_start='2019-12-31', observation_end='2029-12-31')

# Convert to DataFrame
df_gdp = pd.DataFrame(gdp)
df_gdp.index = pd.to_datetime(df_gdp.index)
df_gdp.columns = ['GDP']

# Resample to annual average GDP
df_gdp_annual = df_gdp.resample('YE').mean().reset_index()

# Extract Year
df_gdp_annual['Year'] = df_gdp_annual['index'].dt.year

# Calculate YoY % change in GDP as GDP Growth Rate
df_gdp_annual['GDP_Growth'] = df_gdp_annual['GDP'].pct_change() * 100

# Keep only Year and GDP_Growth columns, drop first NaN row
df_gdp_final = df_gdp_annual[['Year', 'GDP_Growth']].dropna().reset_index(drop=True)

# MERGE DFs

# Merge sequentially on 'Year'
df_macro = df_risk_free.merge(df_cpi_final, on='Year', how='outer') \
    .merge(df_gdp_final, on='Year', how='outer')

# Sort by Year and reset index
df_macro = df_macro.sort_values('Year').reset_index(drop=True)

# Display results
print(df_macro)

In [ ]:
# Assuming df_features_hist and df_macro have 'Year' column as int or compatible dtype

df_X = pd.merge(df_features_hist, df_macro, on='Year', how='left')

print(df_X.info())


In [ ]:
# Define baseline values (replace with actual known values)
baseline_inflation_2019 = 1.8  # example: 1.8%
baseline_gdp_growth_2019 = 2.3  # example: 2.3%
baseline_risk_free_rate_2019 = 1.9  # example: 1.9%

# Fill missing macro values with these baseline values
df_X['Inflation'] = df_X['Inflation'].fillna(baseline_inflation_2019)
df_X['GDP_Growth'] = df_X['GDP_Growth'].fillna(baseline_gdp_growth_2019)
df_X['Risk_Free_Rate'] = df_X['Risk_Free_Rate'].fillna(baseline_risk_free_rate_2019)


In [ ]:
print("First 20 rows of the features dataset (df_X):")
print(df_X.head(20))

print("\nSummary information of the features dataset (df_X):")
print(df_X.info())

print("\nFirst 20 rows of the target dataset (df_y):")
print(df_y.head(20))

print("\nSummary information of the target dataset (df_y):")
print(df_y.info())
